In [ ]:
import pandas as pd
import re
import numpy as np


input_file = "core_articles.csv"
output_file = "core_articles_fixed.csv"


def clean_core_text(text):
    """
    Funkcja czyszcząca tekst:
    1. Usuwa entery i wielokrotne spacje (spłaszcza rozjechane wiersze).
    2. Usuwa tagi HTML/XML.
    3. Usuwa "urwane" tagi specyficzne dla CORE (np. jats).
    """
    if not isinstance(text, str):
        return ""
    
    # 1. Spłaszczanie wierszy (zamiana \n, \r, \t na spację)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 2. Usuwanie standardowych tagów (to co jest pomiędzy < a >)
    # Usuwa <i>, </i>, <p>, <jats:p> jeśli są domknięte
    text = re.sub(r'<[^>]+>', '', text)
    
    # 3. Fix specjalny - usuwanie niedomkniętych tagów jats (np. ...</jats:p)
    ### szuka ciągu zaczynającego się od <jats lub </jats i usuwa go nawet jeśli nie ma końcowego zamknięcia >
    text = re.sub(r'</?jats:[^>\s]*>?', '', text)
    
    return text

print(f"Wczytywanie pliku {input_file}...")
try:
    ### użycie on_bad_lines='skip' na wypadek tragicznie uszkodzonych linii
    #### pandas zazwyczaj radzi sobie z multiline jeśli cudzysłowy są ok
    df = pd.read_csv(input_file, on_bad_lines='skip')
    print(f"Wczytano {len(df):,} wierszy.")
except Exception as e:
    print(f"Błąd krytyczny przy wczytywaniu: {e}")
    df = pd.DataFrame()

if not df.empty:
    # 1. Czyszczenie tekstów (Title, Abstract)
    print("Rozpoczynam czyszczenie tagów i spłaszczanie wierszy...")
    cols_to_clean = ['title', 'abstract', 'keywords']
    
    for col in cols_to_clean:
        if col in df.columns:
            ## konwersja na string i aplikacja funkcji czyszczącej
            df[col] = df[col].astype(str).apply(clean_core_text)
            ## przywracanie pustych stringów jako NaN
            df[col] = df[col].replace(r'^\s*$', np.nan, regex=True)
            df[col] = df[col].replace('nan', np.nan) # usuwanie stringa "nan"

    # 2. Naprawa roku (np. 2009.0 -> 2009)
    if 'year' in df.columns:
        print("Formatowanie kolumny 'year'...")
        ## konwersja na numeryczny, błędy na NaN
        df['year'] = pd.to_numeric(df['year'], errors='coerce')
        ## zapis jako Int64 (typ, który obsługuje NaN, w przeciwieństwie do zwykłego int)
        df['year'] = df['year'].astype('Int64')

    # 3. Deduplikacja
    # sprawdzenie czy kolumna 'creator' istnieje (w fragmencie jej nie było w nagłówku, ale może być w danych)
    dedup_subset = ['title', 'abstract']
    if 'creator' in df.columns:
        dedup_subset.append('creator')
        df['creator'] = df['creator'].astype(str).apply(clean_core_text).replace('nan', np.nan)
    
    print(f"Deduplikacja w oparciu o kolumny: {dedup_subset}...")
    initial_count = len(df)
    # usuwanie wierszy gdzie tytuł lub abstrakt są puste
    # df = df.dropna(subset=['title']) 
    
    df = df.drop_duplicates(subset=dedup_subset, keep='first')
    final_count = len(df)
    
    print(f"Usunięto {initial_count - final_count:,} duplikatów.")
    print(f"Pozostało {final_count:,} unikalnych rekordów.")

    # 4. Zapis
    print(f"Zapisywanie do {output_file}...")
    df.to_csv(output_file, index=False, encoding='utf-8')
    
    # podgląd
    print("\n--- Przykładowe dane po naprawie ---")
    print(df[['title', 'year']].head(3))
    if 'abstract' in df.columns:
        print("\nFragment abstraktu (bez tagów):")
        # pierwszy niepusty abstrakt
        sample_abstract = df['abstract'].dropna().iloc[0] if not df['abstract'].dropna().empty else "Brak abstraktów"
        print(sample_abstract[:200] + "...")

print("\nGotowe. Plik uratowany.")